In [1]:
!pip install langchain langchain-core langchain-community

In [2]:
!pip install pypdf pymupdf sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

sample_doc=Document(
    page_content="Hello Bhoomi",
    metadata={"source":"www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'www.google.com'}, page_content='Hello Bhoomi')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
#Text data-->document
# from langchain_community.document_loaders.text import TextLoader
# loader=TextLoader("./data/Python.txt",encoding="utf-8")
# document=loader.load()

In [7]:
# document

In [8]:
#PDF data-->document (we can use PYPDFLoader for simple pdfs and PYMUPDFLoader for complex pdfs)
# from langchain_community.document_loaders.pdf import PyMuPDFLoader
# pdf_loader=PyMuPDFLoader("./data/research.pdf")
# pdf_doc=pdf_loader.load()
# pdf_doc

In [9]:
###Ingestion pipeline
#Data-->documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

C:\Users\RADHAGOPINATH\AppData\Local\Temp\ipykernel_3876\871006282.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [10]:
def load_all_pdfs():
    folder_path="./data/pdfs"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"): #example research1.pdf
            #complete file path
            pdf_path=os.path.join(folder_path,filename) #it joins as "./data/pdfs/research1.pdf"

            loader=PyPDFLoader(pdf_path)
            doc=loader.load()

            num_docs+=1
            all_docs.extend(doc)

    print(f"total pdfs: ",num_docs)
    print(f"total_pages: ",len(all_docs))
    return all_docs

In [11]:
all_pdf_documents=load_all_pdfs()

total pdfs:  2
total_pages:  25


In [12]:
#chunks
!pip install langchain_text_splitters

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size
    )

    chunked_docs=text_splitter.split_documents(documents)
    return chunked_docs

In [14]:
chunks=split_docs(all_pdf_documents)

In [15]:
len(chunks)

284

In [16]:
#Embedding
from sentence_transformers import SentenceTransformer

In [17]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading model...",self.model_name)

        self.model=SentenceTransformer(self.model_name)
        print("embedding dimensions: ",self.model.get_sentence_embedding_dimension())

    def generate_embedding(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("embedding shape: ",embeddings.shape)
        return embeddings

In [18]:
embedding_manager=EmbeddingManager()

Loading model... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions:  384


C:\Users\RADHAGOPINATH\AppData\Local\Temp\ipykernel_3876\3478893907.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions: ",self.model.get_sentence_embedding_dimension())


In [19]:
import chromadb
import uuid

In [20]:
##Vector Store
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [21]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [23]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

embedding shape:  (284, 384)
total documents added in vector store= 284
docs in collection: 284


In [24]:
##Retrieval Pipeline
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embedding([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [26]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [30]:
rag_retriever.retrieve("What is a rag?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
retrieved 5 documents


[{'id': 'doc_ffd87ef3-501c-4794-980b-8e8e399fdfc7',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'doc_index': 51,
   'subject': '',
   'page': 0,
   'author': '',
   'total_pages': 21,
   'source': './data/pdfs\\research2.pdf',
   'keywords': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'trapped': '/False',
   'creator': 'LaTeX with hyperref',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page_label': '1',
   'content_length': 288,
   'producer': 'pdfTeX-1.40.25',
   'title': '',
   'creationdate': '2024-03-28T00:54:45+00:00'},
  'distance': 0.5337206125259399,
  'similarity_score': 0.46627938747406006,
  'rank': 1},
 {'id': 'doc_335d42

In [31]:
#Integrate with LLM
!pip install python-dotenv

In [32]:
!pip install langchain-openai

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.6 MB 1.5 MB/s eta 0:00:01
   -------------------------------- ------- 1.3/1.6 MB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ----------------------------------- ---- 786.4/874.8 kB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 874.8/874.8 kB 3.3 MB/s eta 0:00:00

   ---------- ----------------------------- 1/4 [tiktoken]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
 

In [34]:
from dotenv import load_dotenv
import os

load_dotenv()

openai_key = os.getenv("OPEN_API_KEY")


In [35]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=openai_key,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [36]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [37]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
retrieved 3 documents


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [38]:
print(answer)

NameError: name 'answer' is not defined

In [39]:
!pip install langchain-groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   -------------------- ------------------- 1/2 [langchain-groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [40]:
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")

In [42]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=groq_key,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [43]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [44]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
retrieved 3 documents


In [45]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me start by recalling the context provided. The context mentions a survey paper that reviews state-of-the-art RAG methods, their evolution through paradigms like naive RAG, and effective RAG frameworks. It also talks about evaluation methods, tasks, datasets, and future directions.

First, I need to define RAG. From the context, RAG stands for Retrieval-Augmented Generation. The paper discusses integrating RAG within Large Language Models (LLMs). So, RAG combines retrieval of external information with generation to enhance the model's responses.

The context mentions different paradigms like naive RAG and effective RAG frameworks. I should explain that RAG typically involves two components: a retriever that fetches relevant documents and a generator that uses those documents to create answers. This helps in providing up-to-date and accurate information by leveraging external data sources.

The survey also covers evaluation methods, w